# V031 combined-architecture synthetic proof
Frozen V029 objective proofs are inherited. New proofs cover protected span3 math, zero-adapter controls, exact combined-model recovery and the private conservative queue. No approved data are read; actual CUDA is never emulated.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,copy,time,gc
import numpy as np
import torch

def run_laplace_convolution_preflight_v031(project_root,output_directory,device='cpu',deep_spec=None):
    P=Path(project_root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False);checks={}
    def check(name,value):
        if not bool(value):raise AssertionError(name)
        checks[name]=True
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
    def equal(a,b):
        if torch.is_tensor(a):return torch.is_tensor(b) and torch.equal(a,b)
        if isinstance(a,np.ndarray):return isinstance(b,np.ndarray) and np.array_equal(a,b)
        if isinstance(a,dict):return isinstance(b,dict) and a.keys()==b.keys() and all(equal(a[k],b[k]) for k in a)
        if isinstance(a,(list,tuple)):return type(a)==type(b) and len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))
        return a==b
    # Preserve/re-run the frozen NLL objective and original engine proofs unchanged.
    inherited_ns={};exec(compile(code(P/'research/laplace_preflight_v029.ipynb'),'laplace_preflight_v029.ipynb','exec'),inherited_ns)
    inherited=inherited_ns['run_laplace_preflight_v029'](P,O/'inherited_v029',device=device)
    for name,value in inherited['checks'].items():check('inherited_v029:'+name,value)
    ns={'__name__':'ventilator_v031_preflight'};sources={}
    for name in ['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb','convolution_components_v024.ipynb','laplace_components_v029.ipynb','laplace_convolution_components_v031.ipynb']:
        text=code(P/'research'/name);sources[name]=hashlib.sha256(text.encode()).hexdigest();exec(compile(text,name,'exec'),ns)
    selected=ns['_sequence_device'](device);ns['seed_sequence'](20261102)
    rng=np.random.default_rng(20263101);n_features=47;steps=np.arange(80,dtype=np.float32)[None,:]
    raw=rng.normal(0.,.2,(8,80,n_features)).astype(np.float32)
    raw[:,:,0]=np.array([5,20,50,5,20,50,5,20],dtype=np.float32)[:,None]
    raw[:,:,1]=np.array([10,20,50,20,50,10,50,10],dtype=np.float32)[:,None]
    raw[:,:,2]=np.broadcast_to(steps*.025,(8,80));raw[:,:,3]=8.+2*np.sin(steps/7.)+.3*np.arange(8)[:,None]
    out=np.broadcast_to(steps>=31,(8,80)).astype(np.int8).copy();raw[:,:,4]=out;raw[:,:,5]=steps;raw[:,:,6]=.025;raw[:,0,6]=0.
    y=(3.+.13*raw[:,:,3]+.07*steps+.1*np.arange(8)[:,None]+.03*rng.normal(size=(8,80))).astype(np.float32)
    names=['R','C','time_step','u_in','u_out','step','dt']+['toy_'+str(i) for i in range(7,n_features)]
    spec=dict(family='lstm',hidden=8,layers=2,dropout=.15,bidirectional=True,residual=True,widths=[8,6],output_concat=True,head_width=8)
    cfg=dict(lr=.0015,batch_size=2,max_epochs=3,patience=0,min_delta=0.,scheduler='replay',frozen_lr_schedule=[.0015,.001,.0005],prediction_batch_size=2,optimizer='Adam',weight_decay=0.,loss='l1')
    prep=ns['fit_preprocessor'](raw[:6],y[:6],out[:6],spec,names);transformed=ns['transform_inputs'](raw,prep)
    tensor=torch.from_numpy(transformed[:2]).to(selected)
    ns['seed_sequence'](20261102);plain=ns['v029_laplace_factory'](spec,n_features,prep).to(selected);base_rng=torch.get_rng_state().clone();base_cuda=torch.cuda.get_rng_state_all() if selected.type=='cuda' else None
    ns['seed_sequence'](20261102);model=ns['v031_conv3_laplace_factory'](spec,n_features,prep).to(selected)
    check('adapter_construction_consumes_no_extra_CPU_RNG',torch.equal(torch.get_rng_state(),base_rng))
    if selected.type=='cuda':check('adapter_construction_consumes_no_extra_CUDA_RNG',equal(torch.cuda.get_rng_state_all(),base_cuda))
    check('all_V029_mean_scale_body_parameters_unchanged',all(torch.equal(v,model.state_dict()[k]) for k,v in plain.state_dict().items()))
    check('new_model_is_V029_subclass',isinstance(model,ns['V029PressureLaplaceRNN']))
    check('only_span3_context_parameter_capacity_added',sum(p.numel() for p in model.parameters())-sum(p.numel() for p in plain.parameters())==3*n_features*n_features)
    check('zero_adapter_is_exact_input_identity',torch.equal(model.adapt_inputs(tensor),tensor))
    plain.eval();model.eval()
    with torch.no_grad():mu0=plain(tensor);b0=plain.v029_scale.clone();mu1=model(tensor);b1=model.v029_scale.clone()
    check('zero_adapter_mean_exact_V029_control',torch.equal(mu0,mu1))
    check('zero_adapter_positive_scale_exact_V029_control',torch.equal(b0,b1))
    # Matching input-gradient requirements make both backbone backward paths identical.
    a=tensor.clone().requires_grad_(True);b=tensor.clone().requires_grad_(True);target=torch.from_numpy(ns['_scaled_targets'](raw[:2],y[:2],prep)).to(selected);mask=torch.from_numpy(out[:2]==0).to(selected)
    plain.train();model.train();plain.zero_grad(set_to_none=True);model.zero_grad(set_to_none=True)
    ns['seed_sequence'](440);p0=plain(a);s0=plain.v029_scale;loss0=ns['laplace_curve_loss_v029'](p0,target,s0,mask)[0];loss0.backward();rng0=torch.get_rng_state().clone();cuda0=torch.cuda.get_rng_state_all() if selected.type=='cuda' else None
    ns['seed_sequence'](440);p1=model(b);s1=model.v029_scale;loss1=ns['laplace_curve_loss_v029'](p1,target,s1,mask)[0];loss1.backward()
    check('zero_adapter_training_mean_and_scale_exact',torch.equal(p0,p1) and torch.equal(s0,s1) and torch.equal(loss0,loss1))
    check('zero_adapter_body_gradients_exact_V029_control',all(torch.equal(v.grad,dict(model.named_parameters())[k].grad) for k,v in plain.named_parameters()))
    check('zero_adapter_input_gradient_exact_control',torch.equal(a.grad,b.grad))
    check('zero_adapter_training_CPU_RNG_exact_control',torch.equal(rng0,torch.get_rng_state()))
    if selected.type=='cuda':check('zero_adapter_training_CUDA_RNG_exact_control',equal(cuda0,torch.cuda.get_rng_state_all()))
    protected=[names.index(n) for n in ['R','C','u_out','time_step','step']]
    check('trainable_adapter_has_nonzero_finite_gradient',model.context_weight.grad is not None and bool((model.context_weight.grad!=0).any()) and bool(torch.isfinite(model.context_weight.grad).all()))
    check('protected_adapter_output_rows_have_zero_gradient',bool((model.context_weight.grad[:,protected,:]==0).all()))
    # Independently invoke the frozen V024 implementation on learned (nonzero) weights.
    reference=ns['V024TemporalResidualRNN'](spec,n_features,names,3).to(device=selected,dtype=torch.float64)
    model=model.to(dtype=torch.float64)
    with torch.no_grad():weights=torch.randn_like(model.context_weight)*.01;model.context_weight.copy_(weights);reference.context_weight.copy_(weights)
    q=torch.randn(3,80,n_features+1,dtype=torch.float64,device=selected);q[...,-1]=torch.arange(3,dtype=torch.float64,device=selected)[:,None]
    q0=q.clone().requires_grad_(True);q1=q.clone().requires_grad_(True);r0=reference.adapt_inputs(q0);r1=model.adapt_inputs(q1);probe=torch.randn_like(r0)
    grad0=torch.autograd.grad((r0*probe).sum(),(q0,reference.context_weight));grad1=torch.autograd.grad((r1*probe).sum(),(q1,model.context_weight))
    check('learned_adapter_exact_frozen_V024_forward',torch.equal(r0,r1))
    check('learned_adapter_exact_frozen_V024_input_weight_gradients',all(torch.equal(x,z) for x,z in zip(grad0,grad1)))
    check('learned_adapter_preserves_R_C_phase_time_step_and_jointcode',torch.equal(r1[...,protected],q[...,protected]) and torch.equal(r1[...,-1],q[...,-1]))
    changed=q.clone();changed[1:,:,:n_features]+=100.
    check('adapter_has_no_cross_breath_context',torch.equal(model.adapt_inputs(changed)[0],r1[0]))
    local=q.clone();local[:,40,3]+=1.;delta=model.adapt_inputs(local)-r1
    allowed=torch.zeros(80,dtype=torch.bool,device=selected);allowed[39:42]=True
    check('span3_context_locality_exact',bool((delta[:,~allowed,:]==0).all()) and bool((delta[:,allowed,:]!=0).any()))
    ramp=torch.arange(80,dtype=torch.float64,device=selected).reshape(1,80,1)
    check('temporal_padding_never_wraps',model.shifted(ramp,1)[0,-1,0]==0 and model.shifted(ramp,-1)[0,0,0]==0)
    model.eval();q32=tensor.to(dtype=torch.float64)
    with torch.no_grad():first=model(q32);scale_first=model.v029_scale.clone();again=model(q32);scale_again=model.v029_scale.clone()
    check('both_mean_and_scale_memory_reset_between_calls',torch.equal(first,again) and torch.equal(scale_first,scale_again))
    del model,plain,reference,a,b,p0,p1,s0,s1,loss0,loss1,q0,q1,r0,r1,grad0,grad1;gc.collect()
    if selected.type=='cuda':torch.cuda.empty_cache()
    # A frozen zero adapter must preserve complete Adam/clip/dropout training, not only its first forward.
    ns['seed_sequence'](999);control=ns['v029_laplace_factory'](spec,n_features,prep).to(selected)
    ns['seed_sequence'](999);frozen=ns['v031_frozen_conv3_laplace_factory'](spec,n_features,prep).to(selected)
    opt0=ns['_optimizer'](control,ns['_train_configuration'](cfg));opt1=ns['_optimizer'](frozen,ns['_train_configuration'](cfg))
    control.train();frozen.train()
    for iteration in range(3):
        xx=torch.from_numpy(transformed[2*iteration:2*iteration+2]).to(selected);tt=torch.from_numpy(ns['_scaled_targets'](raw[2*iteration:2*iteration+2],y[2*iteration:2*iteration+2],prep)).to(selected);mm=torch.from_numpy(out[2*iteration:2*iteration+2]==0).to(selected)
        ns['seed_sequence'](800+iteration);opt0.zero_grad(set_to_none=True);mean0=control(xx);scale0=control.v029_scale;loss0=ns['laplace_curve_loss_v029'](mean0,tt,scale0,mm)[0];loss0.backward();ns['nn'].utils.clip_grad_norm_(control.parameters(),1.);opt0.step()
        ns['seed_sequence'](800+iteration);opt1.zero_grad(set_to_none=True);mean1=frozen(xx);scale1=frozen.v029_scale;loss1=ns['laplace_curve_loss_v029'](mean1,tt,scale1,mm)[0];loss1.backward();ns['nn'].utils.clip_grad_norm_(frozen.parameters(),1.);opt1.step()
        check('frozen_zero_adapter_step'+str(iteration)+'_mean_scale_loss_exact',torch.equal(mean0,mean1) and torch.equal(scale0,scale1) and torch.equal(loss0,loss1))
        check('frozen_zero_adapter_step'+str(iteration)+'_all_V029_parameters_exact',all(torch.equal(p,dict(frozen.named_parameters())[k]) for k,p in control.named_parameters()))
        check('frozen_zero_adapter_step'+str(iteration)+'_Adam_moments_exact',all(equal(opt0.state[p],opt1.state[dict(frozen.named_parameters())[k]]) for k,p in control.named_parameters()))
    check('frozen_adapter_stays_zero',bool((frozen.context_weight==0).all()) and frozen.context_weight.grad is None)
    del control,frozen,opt0,opt1,mean0,mean1,scale0,scale1,loss0,loss1;gc.collect()
    if selected.type=='cuda':torch.cuda.empty_cache()
    common=dict(model_spec=spec,train_cfg=cfg,seed=20261102,feature_names=names,train_diagnostic_breaths=4,
        device=device,model_factory=ns['v031_conv3_laplace_factory'],loss_kind='laplace_curve',arm='v031_synthetic',source_fingerprints=sources)
    class Interrupted(RuntimeError):pass
    def interrupt(record):
        if record['epoch']==1:raise Interrupted('Saved epoch-boundary synthetic interruption')
    summaries={}
    for mode in ['validate','fixed']:
        full=O/(mode+'-full.pt');resume_path=O/(mode+'-resume.pt')
        def run(path,resume=False,on_epoch=None,**overrides):
            kw=dict(common,checkpoint_path=path,resume=resume,on_epoch=on_epoch);kw.update(overrides)
            if mode=='validate':return ns['fit_validate_v031'](raw[:6],y[:6],out[:6],raw[6:],y[6:],out[6:],**kw)
            kw.pop('train_diagnostic_breaths');return ns['fit_fixed_v031'](raw[:6],y[:6],out[:6],lr_schedule=cfg['frozen_lr_schedule'],**kw)
        full_report=run(full)
        try:run(resume_path,on_epoch=interrupt)
        except Interrupted:checks[mode+'_interrupt_after_authoritative_checkpoint']=True
        else:raise AssertionError('No synthetic interruption')
        resumed_report=run(resume_path,resume=True)
        aa=torch.load(ns['_last_path'](full),map_location='cpu',weights_only=False);bb=torch.load(ns['_last_path'](resume_path),map_location='cpu',weights_only=False)
        for field in ['state_dict','optimizer_state','rng_state','scheduler_state','preprocess','fingerprint','model_spec','config','epoch','optimizer_steps']:
            check(mode+'_exact_resume_'+field,equal(aa[field],bb[field]))
        fields=['lr_used','training_online_inspiratory_mae','training_online_objective_scaled','training_online_laplace_NLL_scaled','training_online_increment_mae','training_online_scale_mean_native']
        if mode=='validate':fields+=['validation_inspiratory_mae','best_val_mae','best_epoch']
        check(mode+'_exact_epoch_metrics',all(all(equal(a[f],b[f]) for f in fields) for a,b in zip(full_report['epochs'],resumed_report['epochs'])))
        options=dict(device=device,batch_size=2,source_fingerprints=sources,expected_arm=common['arm'],expected_loss_kind='laplace_curve')
        mu,scale=ns['predict_checkpoint_laplace_v031'](full,raw[6:],return_scale=True,**options)
        rm,rb=ns['predict_checkpoint_laplace_v031'](resume_path,raw[6:],return_scale=True,**options)
        check(mode+'_native_mean_and_scale_exact_resume',np.array_equal(mu,rm) and np.array_equal(scale,rb))
        check(mode+'_native_prediction_finite_positive_scale',mu.shape==scale.shape==(2,80) and np.isfinite(mu).all() and np.isfinite(scale).all() and (scale>0).all())
        check(mode+'_truthful_primary_objective_metadata',full_report['loss_kind']=='laplace_curve' and full_report['actual_training_objective']=='laplace_curve' and aa['v029']['objective']['loss_kind']=='laplace_curve' and aa['actual_training_objective']=='laplace_curve')
        check(mode+'_truthful_architecture_metadata',full_report['v031']['architecture']==ns['V031_ARCHITECTURE'] and aa['v031']['architecture']==ns['V031_ARCHITECTURE'] and aa['fingerprint']['v031']==aa['v031'] and aa['actual_model_architecture']=='protected_span3_before_v029')
        if mode=='validate':
            native=ns['inspiratory_mae'](y[6:],mu,out[6:])
            check('native_MAE_best_checkpoint_selection',full_report['best_epoch']==int(np.argmin([r['validation_inspiratory_mae'] for r in full_report['epochs']]))+1 and abs(native-full_report['validation_mae'])<1e-15)
        for label,override in [('arm',dict(arm='v029_changed')),('loss',dict(loss_kind='curve_l1')),('source',dict(source_fingerprints=dict(sources,changed_source='0'*64)))]:
            try:run(resume_path,resume=True,**override)
            except ValueError:checks[mode+'_reject_changed_'+label]=True
            else:raise AssertionError('Changed identity accepted: '+label)
        before=copy.deepcopy(ns['V029_OBJECTIVE']);ns['V029_OBJECTIVE']['scale_floor_target_scaled']=.002
        try:
            try:run(resume_path,resume=True)
            except ValueError:checks[mode+'_reject_changed_objective_configuration']=True
            else:raise AssertionError('Changed objective accepted')
        finally:ns['V029_OBJECTIVE'].clear();ns['V029_OBJECTIVE'].update(before)
        corrupted=O/(mode+'-tampered-architecture.pt');payload=torch.load(full,map_location='cpu',weights_only=False);payload['v031']['architecture']['activation']='tampered';torch.save(payload,corrupted)
        try:ns['load_checkpoint_v031'](corrupted,device=device,source_fingerprints=sources)
        except ValueError:checks[mode+'_own_loader_rejects_architecture_tamper_before_V029']=True
        else:raise AssertionError('Own architecture guard ignored')
        previous_architecture=copy.deepcopy(ns['V031_ARCHITECTURE']);ns['V031_ARCHITECTURE']['span']=7
        try:
            try:run(resume_path,resume=True)
            except ValueError:checks[mode+'_reject_changed_architecture_recipe']=True
            else:raise AssertionError('Changed architecture recipe accepted')
        finally:ns['V031_ARCHITECTURE'].clear();ns['V031_ARCHITECTURE'].update(previous_architecture)
        summaries[mode]=dict(executed_epochs=full_report['executed_epochs'],checkpoint_sha256=sha(full),last_sha256=sha(ns['_last_path'](full)))
        del aa,bb;gc.collect()
    workflow=ns['v031_workflow_namespace'](P);policy_path=P/'research/pruning_policy_v025.json';policy_sha=sha(policy_path)
    job=dict(id='v031_queue_synthetic_representative',variant='conv3_laplace_curve',fit_args=(raw[:6],y[:6],out[:6],raw[6:],y[6:],out[6:],spec,cfg,20261102,O/'queue-model.pt'),fit_kwargs=dict(feature_names=names,device=device,model_factory=ns['v031_conv3_laplace_factory'],loss_kind='laplace_curve',arm='v031_queue_synthetic',source_fingerprints=sources,train_diagnostic_breaths=4))
    queued=workflow['run_training_queue_v025']([job],project_root=P,policy_path=policy_path,expected_policy_sha256=policy_sha,state_path=O/'queue-state.json')
    check('V025_queue_executes_full_unfamiliar_objective',queued[0]['status']=='completed' and queued[0]['report']['executed_epochs']==3)
    check('V025_identity_truthful_objective',queued[0]['identity']['cohort']['objective']==ns['V029_OBJECTIVE'])
    check('V025_identity_truthful_architecture',queued[0]['identity']['cohort']['architecture']==ns['V031_ARCHITECTURE'])
    decisions=[json.loads(l) for l in (O/'queue-model.pruning.jsonl').read_text().splitlines()]
    check('unfamiliar_objective_not_aggressively_pruned',len(decisions)==3 and all(v['action']=='continue' for v in decisions))
    pred=workflow['predict_completed_v025'](queued[0],raw[6:],device=device,batch_size=2)
    check('private_workflow_completed_prediction_truthful_loader',pred.shape==(2,80) and np.isfinite(pred).all())
    check('original_V008_trainer_binding_unchanged',ns['fit_validate_v008'] is not ns['fit_validate_v031'])
    profiles=[]
    if deep_spec is not None:
        ns['seed_sequence'](20261102);baseline=ns['v029_laplace_factory'](deep_spec,n_features,prep).to(selected)
        ns['seed_sequence'](20261102);deep=ns['v031_conv3_laplace_factory'](deep_spec,n_features,prep).to(selected)
        baseline.eval();deep.eval()
        with torch.no_grad():mu0=baseline(tensor);b0=baseline.v029_scale.clone();mu1=deep(tensor);b1=deep.v029_scale.clone()
        check('full_deep_zero_adapter_mean_and_scale_exact',torch.equal(mu0,mu1) and torch.equal(b0,b1) and mu1.shape==(2,80))
        check('full_deep_scale_positive_finite',bool((b1>0).all()) and bool(torch.isfinite(b1).all()))
        del baseline,deep;gc.collect()
        timing_target=ns['_scaled_targets'](raw[:1],y[:1],prep)
        check('full_model_timing_helper_exercised_on_CPU_and_CUDA',timing_target.shape==(1,80) and timing_target.dtype==np.float32 and np.isfinite(timing_target).all())
        if selected.type=='cuda':
            torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats();batch=256
            tx=torch.from_numpy(np.repeat(transformed[:1],batch,axis=0)).to(selected);ty=torch.from_numpy(np.repeat(timing_target,batch,axis=0)).to(selected);tm=torch.from_numpy(np.repeat(out[:1]==0,batch,axis=0)).to(selected)
            for name,factory in [('laplace_baseline',ns['v029_laplace_factory']),('conv3_laplace',ns['v031_conv3_laplace_factory'])]:
                ns['seed_sequence'](20261102);m=factory(deep_spec,n_features,prep).to(selected);opt=torch.optim.Adam(m.parameters(),lr=.0015);times=[]
                for repeat in range(4):
                    torch.cuda.synchronize();started=time.perf_counter();opt.zero_grad(set_to_none=True);mean=m(tx)
                    loss=ns['laplace_curve_loss_v029'](mean,ty,m.v029_scale,tm)[0]
                    loss.backward();ns['nn'].utils.clip_grad_norm_(m.parameters(),1.);opt.step();torch.cuda.synchronize()
                    if repeat>0:times.append(time.perf_counter()-started)
                profiles.append(dict(name=name,batch_size=batch,median_step_seconds=float(np.median(times)),peak_allocated_bytes=int(torch.cuda.max_memory_allocated())))
                del m,opt,mean,loss;gc.collect();torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
            profiles[1]['step_time_ratio_to_baseline']=profiles[1]['median_step_seconds']/profiles[0]['median_step_seconds']
            check('full_model_CUDA_efficiency_bounded',profiles[1]['step_time_ratio_to_baseline']<1.7 and profiles[1]['peak_allocated_bytes']<10*1024**3)
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),checks=checks,check_count=len(checks),source_code_hashes=sources,
        inherited_v029_proof_sha256=sha(O/'inherited_v029/proof.json'),actual_cuda=selected.type=='cuda',exact_resume=True,native_inference=True,
        zero_adapter_control=True,protected_channels=True,raw_approved_csv_reads=0,approved_data_fits=0,synthetic_only=True,
        objective=copy.deepcopy(ns['V029_OBJECTIVE']),architecture=copy.deepcopy(ns['V031_ARCHITECTURE']),summaries=summaries,efficiency_profiles=profiles,
        runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,device=device))
    (O/'proof.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result,indent=2));return result


{
  "status": "passed",
  "created_utc": "2026-11-08T17:01:33.874216+00:00",
  "checks": {
    "inherited_v029:actual_requested_device": true,
    "inherited_v029:frozen_unit_scale_matches_original_curve_loss": true,
    "inherited_v029:frozen_unit_scale_matches_original_mean_gradient": true,
    "inherited_v029:expired_mean_output_has_zero_objective_gradient": true,
    "inherited_v029:expired_scale_output_has_zero_objective_gradient": true,
    "inherited_v029:analytic_scale_gradient_correct": true,
    "inherited_v029:expired_pressure_does_not_change_likelihood_or_curve": true,
    "inherited_v029:only_adjacent_inspiratory_pairs_penalized": true,
    "inherited_v029:empty_adjacent_pair_term_finite_zero": true,
    "inherited_v029:varying_positive_scale_likelihood_exact": true,
    "inherited_v029:nonpositive_scale_rejected": true,
    "inherited_v029:scale_head_initialization_consumes_no_CPU_RNG": true,
    "inherited_v029:all_original_mean_parameters_exact_at_initialization": true,